In [1]:
import shutil, os, sys

src = '/kaggle/input/datasets/jasmine1912/safebite-backend'
dst = '/kaggle/working/safebite-backend'

if not os.path.exists(dst):
    shutil.copytree(src, dst)

sys.path.insert(0, dst)

for mod in list(sys.modules):
    if mod == 'app' or mod.startswith('app.'):
        del sys.modules[mod]

from app.ai.craving_vae import craving_text_to_vector, rank_swaps_vae, SWAP_CATALOG

print("Import successful")

Import successful


In [2]:
ALLERGEN_ALIASES = {
    "milk": ["milk", "casein", "whey", "lactose", "ghee", "caseinate"],
    "peanut": ["peanut", "peanuts", "groundnut"],
    "wheat": ["wheat", "flour", "gluten"],
    "egg": ["egg", "eggs", "albumin"],
    "soy": ["soy", "soybean", "soya"],
}

def compute_verdict(text, user_allergens):
    text = text.lower()
    flagged = []
    for allergen in user_allergens:
        aliases = ALLERGEN_ALIASES.get(allergen, [allergen])
        if any(alias in text for alias in aliases):
            flagged.append(allergen)
    if len(text.strip()) < 8:
        return "unclear", []
    if flagged:
        return "flagged", flagged
    return "safe", []

In [3]:
ALLERGEN_TAG_OVERRIDES = {
    "sw8": ["soy"],
    "sw9": ["milk"],
    "sw10": ["tree nut", "soy"],
    "sw12": ["soy"],
}
for item in SWAP_CATALOG:
    item["allergen_tags"] = ALLERGEN_TAG_OVERRIDES.get(item["id"], [])

from app.ai import craving_vae as cv_module
_original_rank = cv_module.rank_swaps_vae

def rank_swaps_vae(query_vec, limit=3, exclude_allergens=None):
    exclude_allergens = exclude_allergens or set()
    candidates = _original_rank(query_vec, limit=len(SWAP_CATALOG))
    safe = [
        c for c in candidates
        if not (set(next(item["allergen_tags"] for item in SWAP_CATALOG if item["id"] == c["id"])) & exclude_allergens)
    ]
    return safe[:limit]

print("Patch applied")

Patch applied


In [4]:
def run_pipeline(label_text, user_allergens, case_name=""):
    print(f"\n{'='*50}")
    print(f"CASE: {case_name}")
    print(f"{'='*50}")

    print(f"[Stage 1] OCR read: '{label_text}'")

    verdict, flagged = compute_verdict(label_text, user_allergens)
    print(f"[Stage 2] Verdict: {verdict.upper()} ({', '.join(flagged) if flagged else 'none'})")

    if verdict in ("flagged", "unclear"):
        craving_vec = craving_text_to_vector(label_text)
        swaps = rank_swaps_vae(craving_vec, limit=3, exclude_allergens=set(user_allergens))
        top_swap = swaps[0] if swaps else None
        print(f"[Stage 3] Top swap: '{top_swap['name']}'" if top_swap else "[Stage 3] No safe swap found")
    else:
        top_swap = None
        print("[Stage 3] No swap needed — item is safe.")

    print(f"\n--- Full Trace ---")
    print(f"OCR text  → {label_text}")
    print(f"Verdict   → {verdict.upper()}")
    print(f"Swap      → {top_swap['name'] if top_swap else 'N/A'}")

In [5]:
user_allergens = ["milk"]

run_pipeline(
    "ingredients: milk, sugar, wheat flour, salt",
    user_allergens,
    case_name="Flagged scan (contains milk)"
)

run_pipeline(
    "ingredients: rice, water, sea salt, black pepper",
    user_allergens,
    case_name="Safe scan (no allergens)"
)

run_pipeline(
    "xyz",
    user_allergens,
    case_name="Unclear scan (OCR failed / too short)"
)


CASE: Flagged scan (contains milk)
[Stage 1] OCR read: 'ingredients: milk, sugar, wheat flour, salt'
[Stage 2] Verdict: FLAGGED (milk)
[Stage 3] Top swap: 'Popped Sorghum'

--- Full Trace ---
OCR text  → ingredients: milk, sugar, wheat flour, salt
Verdict   → FLAGGED
Swap      → Popped Sorghum

CASE: Safe scan (no allergens)
[Stage 1] OCR read: 'ingredients: rice, water, sea salt, black pepper'
[Stage 2] Verdict: SAFE (none)
[Stage 3] No swap needed — item is safe.

--- Full Trace ---
OCR text  → ingredients: rice, water, sea salt, black pepper
Verdict   → SAFE
Swap      → N/A

CASE: Unclear scan (OCR failed / too short)
[Stage 1] OCR read: 'xyz'
[Stage 2] Verdict: UNCLEAR (none)
[Stage 3] Top swap: 'Coconut Yogurt Bark'

--- Full Trace ---
OCR text  → xyz
Verdict   → UNCLEAR
Swap      → Coconut Yogurt Bark
